In [1]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [2]:
# display = print
# HTML = lambda x: x

In [3]:
original_distribution = pd.read_csv(
    "~/Box/dsi-core/11th-hour/good-food-purchasing/CONFIDENTIAL_GFPP Product Attribute List_8.26.25.csv",
    dtype=str,
)[
    [
        "Level of Processing",
    ]
].dropna(
    subset="Level of Processing"
)
original_distribution["Level of Processing"] = original_distribution["Level of Processing"].map(
    {
        "Whole/Minimally Processed": 1,
        "Culinary Ingredient": 2,
        "Moderately Processed": 3,
        "Ultra-Processed": 4,
    }
)
original_distribution = original_distribution[original_distribution["Level of Processing"].notna()]
original_distribution["Level of Processing"] = original_distribution["Level of Processing"].astype(int)
original_distribution = original_distribution["Level of Processing"].value_counts()
original_distribution

Level of Processing
4    44109
1    30744
3     3653
2     3532
Name: count, dtype: int64

In [4]:
df = pd.read_csv("~/Box/dsi-core/11th-hour/good-food-purchasing/cgfp-training-try1/cgfp-test-results.csv")

In [5]:
wrong_distribution = df["Level of Processing"].value_counts()
wrong_distribution

Level of Processing
4    40676
1    27312
3      221
2      100
Name: count, dtype: int64

In [6]:
weights = (original_distribution / original_distribution.loc[2]) / (wrong_distribution / wrong_distribution.loc[2])
weights

Level of Processing
4    0.030702
1    0.031870
3    0.467990
2    1.000000
Name: count, dtype: float64

In [7]:
dict(weights)

{4: np.float64(0.03070211389020458),
 1: np.float64(0.031870301556245983),
 3: np.float64(0.46799014056358673),
 2: np.float64(1.0)}

In [8]:
df["weight"] = df["Level of Processing"].map(dict(weights))

In [9]:
df

,index,Food Product Category,Primary Food Product Category,Level of Processing,message,prob1,prob2,prob3,prob4,weight
0,54722,Condiments & Snacks,Condiments & Snacks,4,Super Bakery\n\n9494- LET`S CELEBRATE CUPCAKE,3.653482e-08,1.046740e-08,1.725783e-08,1.000000e+00,0.030702
1,85358,Vegetables,Vegetables,1,\n\nBROCCOLI FLORETS FRZ (452040),9.999991e-01,4.363462e-09,6.023574e-08,6.893609e-07,0.031870
2,34542,Milk & Dairy,Milk & Dairy,4,Clover Farms Dairy\n\nCREAM HEAVY 40% 1 QT (24...,8.716333e-04,4.199764e-02,9.558618e-01,1.268219e-03,0.030702
3,15676,Condiments & Snacks,Condiments & Snacks,4,Tropical Nut & Fruits (DSD)/Tropical Nut & Fru...,1.605228e-09,3.581748e-10,2.789468e-10,1.000000e+00,0.030702
4,43704,Tree Nuts & Seeds,Tree Nuts & Seeds,4,"MADEIRA FARMS\n\nPEANUT BUTTER, SS CUP",2.139598e-06,8.462275e-06,5.340276e-02,9.465865e-01,0.030702
...,...,...,...,...,...,...,...,...,...,...
68304,55936,Vegetables,Vegetables,1,RELFRSH\n\nTOMATO ROMA UTILITY FRESH,9.999650e-01,1.057134e-07,4.181041e-07,2.237950e-07,0.031870
68305,68587,Grain Products,Grain Products,1,\n\nPASTA ROTINI WG,9.902518e-01,8.567356e-03,1.159466e-03,2.123635e-05,0.031870
68306,8015,Chicken,Chicken,1,"PATUXENT\n\nCHICKEN, BREAST SINGLE-LOBE 4 OZ B...",9.999992e-01,4.363462e-09,2.536017e-07,2.102432e-07,0.031870
68307,90118,Condiments & Snacks,Condiments & Snacks,4,MAJOR\nMAJOR\nGRAVY MIX BROWN LS (2688500),5.315775e-08,1.209865e-06,3.466321e-07,9.999983e-01,0.030702


In [10]:
len(df)

68309

In [11]:
df["total_prob"] = df["prob1"] + df["prob2"] + df["prob3"] + df["prob4"]
df[df["total_prob"] < 0.99].sort_values("total_prob", ascending=False)

,index,Food Product Category,Primary Food Product Category,Level of Processing,message,prob1,prob2,prob3,prob4,weight,total_prob
1726,5901,Beverages,Beverages,4,Pepsi\n\n28OZ PL GAT G2 GRP 1/15,0.111571,0.046510,7.132511e-03,0.824406,0.030702,0.989620
22823,3575,Beverages,Beverages,4,GATORADE\n\nIsotonices Bottle Ss Lemon Lime Wi...,0.001301,0.463066,3.888897e-06,0.524723,0.030702,0.989093
29849,5555,Beverages,Beverages,4,Pepsi\n\n1G BIB DOL APL 100% 3/1POS NEW,0.006618,0.000226,1.127495e-05,0.982232,0.030702,0.989088
42633,88241,Beverages,Beverages,4,Propel\nPropel\nDrink Fitness Water Berry,0.744427,0.000988,8.716338e-04,0.241680,0.030702,0.987966
6270,80743,Beverages,Beverages,4,Coca Cola\nVITAMINWATER\n16.9Z PT 6P HC VW ZRO...,0.000177,0.000007,1.353663e-06,0.987384,0.030702,0.987569
29278,4442,Beverages,Beverages,4,Sparkling ICE\n\nWATER SPRKLG STRAWB WTRMLN 12...,0.490111,0.231512,1.214864e-03,0.262338,0.030702,0.985175
29048,4323,Beverages,Beverages,4,Pepsi\n\n2L PL CRSH STRW 1/8,0.001151,0.000027,5.979768e-07,0.982933,0.030702,0.984112
53503,52146,Beverages,Beverages,4,"Pedialyte\n\nPedialyte Electrolyte Solution, A...",0.000359,0.036571,3.164013e-04,0.943179,0.030702,0.980425
33025,51236,Beverages,Beverages,4,COCA COLA BOTTLERS SALES&SVC\n\nWATER BTLD POW...,0.456539,0.000324,4.972113e-05,0.517327,0.030702,0.974240
10921,80742,Beverages,Beverages,4,Coca Cola\nVITAMINWATER\n16.9Z PT 6P HC VW ZRO...,0.000538,0.000014,2.493415e-06,0.973500,0.030702,0.974055


In [12]:
def confusion_count(df, normalize=True):
    matrix = np.zeros((4, 4), dtype=float)
    for i in range(4):
        selected = df[df["Level of Processing"] == i + 1]
        denominator = np.sum(selected["weight"])
        bests = np.argmax(selected[["prob1", "prob2", "prob3", "prob4"]].to_numpy(), axis=1)
        for j in range(4):
            if normalize:
                if denominator != 0:
                    matrix[i, j] = np.sum(selected[bests == j]["weight"]) / denominator
            else:
                matrix[i, j] = np.sum(selected[bests == j]["weight"])
    return matrix

def precision(matrix):
    return np.diagonal(matrix) / matrix.sum(axis=0)

In [13]:
def confusion_HTML(matrix, decimal_places=3, colorize=True):
    display(HTML(f"""

<table>
  <tr><td style="text-align: right;"><b>model predicts</b></td>{
      ''.join(f'<td style="text-align: right;"><b>{i}</b></td>' for i in range(1, 5))
  }
  {''.join(
      f'<tr><td style="text-align: right;"><b>{"truth is " if j == 1 else ""}{j}</b></td>'
      + ''.join(f'<td style="text-align: right; background-color: #{
          int(round(256 * (1 - matrix[j - 1, i - 1]))) if colorize else 255:02x
      }ffff">{
          round(matrix[j - 1, i - 1], decimal_places)
      }</td>' for i in range(1, 5))
      + '</tr>' for j in range(1, 5))}
</table>

"""))

In [14]:
display(HTML("Count in each confusion matrix entry, weighted to reproduce the original distribution of NOVA categories:"))
confusion_HTML(confusion_count(df, normalize=False), colorize=False)
display(HTML("<br>"))

display(HTML("Same, normalized by truth category (for computing the accuracy):"))
confusion_HTML(confusion_count(df))
display(HTML("<br>"))

display(HTML(f"Precision when model predicts {
    ' '.join(f'<span style="margin-left: 10px;"><b>{i + 1}</b>: {x * 100:.0f}%</span>'
            for i, x in enumerate(precision(confusion_count(df, normalize=False))))
}"))

model predicts,1,2,3,4
truth is 1,711.058,46.053,56.761,56.57
2,2.0,96.0,0.0,2.0
3,2.34,3.276,74.878,22.932
4,35.307,45.685,94.286,1073.561


model predicts,1,2,3,4
truth is 1,0.817,0.053,0.065,0.065
2,0.02,0.96,0.0,0.02
3,0.023,0.032,0.724,0.222
4,0.028,0.037,0.075,0.86


In [15]:
df["Food Product Category"].value_counts()

Food Product Category
Condiments & Snacks      16254
Vegetables                8552
Meals                     7270
Fruit                     6469
Grain Products            5665
Beverages                 5344
Roots & Tubers            3228
Chicken                   2495
Beef                      2417
Pork                      1779
Cheese                    1628
Milk & Dairy              1085
Turkey, Other Poultry     1060
Milk                       899
Yogurt                     841
Seafood                    754
Legumes                    442
Eggs                       436
Rice                       434
Tree Nuts & Seeds          433
Meat                       423
Fish (Wild)                229
Fish (Farm-Raised)         106
Produce                     41
Butter                       9
Fish (Farm-raised)           5
Fish (Wild)                  2
Fish (Farmed-Raised)         2
Chicken                      1
meals                        1
Meals                        1
Fish (Wild-Caught

In [16]:
consolidation = {
    "Roots & Tubers": "Roots, Tubers, Legumes & Rice",
    "Milk & Dairy": "Milk, Dairy & Eggs",
    "Cheese": "Milk, Dairy & Eggs",
    "Milk": "Milk, Dairy & Eggs",
    "Yogurt": "Milk, Dairy & Eggs",
    "Legumes": "Roots, Tubers, Legumes & Rice",
    "Eggs": "Milk, Dairy & Eggs",
    "Fish (Wild)": "Seafood",
    "Fish (Farm-Raised)": "Seafood",
    "Produce": "Vegetables",
    "Butter": "Milk, Dairy & Eggs",
    "Fish (Farm-raised)": "Seafood",
    "Fish (Farmed-Raised)": "Seafood",
    "Fish (Wild) ": "Seafood",
    "meals": "Meals",
    "Meals ": "Meals",
    "Fish (Wild-Caught)": "Seafood",
    "Milk & Dairy ": "Milk, Dairy & Eggs",
    "Chicken ": "Chicken",
    "Rice": "Roots, Tubers, Legumes & Rice",
    "Tree Nuts & Seeds": "Roots, Tubers, Legumes & Rice",
    "fruit": "Fruit",
    "Chicken": "Meat",
    "Beef": "Meat",
    "Pork": "Meat",
    "Chicken ": "Meat",
    "Turkey, Other Poultry": "Meat",
    "Pork": "Meat",
    "pork": "Meat",
}

df["category"] = df["Food Product Category"].map(lambda x: consolidation.get(x, x))
categories = list(df["category"].value_counts().index)
df["category"].value_counts()

category
Condiments & Snacks              16254
Vegetables                        8593
Meat                              8176
Meals                             7272
Fruit                             6470
Grain Products                    5665
Beverages                         5344
Milk, Dairy & Eggs                4899
Roots, Tubers, Legumes & Rice     4537
Seafood                           1099
Name: count, dtype: int64

In [17]:
for category in categories:
    selected = df.query(f"category == {category!r}")
    counts = ", ".join(f"{np.count_nonzero(selected["Level of Processing"] == i)}" for i in range(1, 5))

    display(HTML(f"<b>Category:</b> {category} has {len(selected)} instances ({counts} in each row)"))
    confusion_HTML(confusion_count(selected))
    display(HTML(f"Precision when model predicts {
        ' '.join(f'<span style="margin-left: 10px;"><b>{i + 1}</b>: {x * 100:.0f}%</span>'
        for i, x in enumerate(precision(confusion_count(selected, normalize=False))))
    }"))

    display(HTML("<br>"))

model predicts,1,2,3,4
truth is 1,0.291,0.667,0.028,0.015
2,0.011,0.968,0.0,0.022
3,0.014,0.042,0.5,0.444
4,0.005,0.086,0.063,0.846


model predicts,1,2,3,4
truth is 1,0.919,0.005,0.054,0.022
2,0.0,0.0,0.0,0.0
3,0.091,0.0,0.727,0.182
4,0.145,0.0,0.556,0.299


model predicts,1,2,3,4
truth is 1,0.784,0.002,0.032,0.182
2,0.0,0.0,0.0,0.0
3,0.2,0.0,0.4,0.4
4,0.052,0.002,0.05,0.896


model predicts,1,2,3,4
truth is 1,0.114,0.009,0.246,0.632
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0
4,0.011,0.001,0.042,0.946


model predicts,1,2,3,4
truth is 1,0.865,0.004,0.082,0.049
2,0.0,0.0,0.0,0.0
3,0.083,0.083,0.667,0.167
4,0.119,0.013,0.369,0.498


model predicts,1,2,3,4
truth is 1,0.636,0.107,0.065,0.192
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.333,0.667
4,0.002,0.007,0.012,0.98


model predicts,1,2,3,4
truth is 1,0.803,0.012,0.003,0.181
2,0.0,0.0,0.0,0.0
3,0.0,0.2,0.2,0.6
4,0.118,0.026,0.011,0.845


model predicts,1,2,3,4
truth is 1,0.825,0.007,0.099,0.068
2,0.143,0.857,0.0,0.0
3,0.016,0.0,0.935,0.048
4,0.025,0.023,0.247,0.705


model predicts,1,2,3,4
truth is 1,0.881,0.006,0.092,0.021
2,0.0,0.0,0.0,0.0
3,0.0,0.053,0.947,0.0
4,0.074,0.051,0.394,0.481


model predicts,1,2,3,4
truth is 1,0.712,0.002,0.199,0.087
2,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0
4,0.079,0.002,0.176,0.743
